# Choosing the AOI

The Kampar box came back **89.6% forest / 8.4% agriculture**, which is too lopsided for a strong change story and bad for class balance. Here, I score a handful of candidate boxes across the Riau clearing frontier on their land-cover mix, then lock the best one.

**Want:** forest roughly **40–75%** with the **highest agriculture share** I can get (real forest-plantation conversion to detect), while keeping the box a similar size (~3,000 km²) so tiling stays consistent.

In [62]:
!pip install -q earthengine-api

In [63]:
import ee
from IPython.display import Image as IPyImage, display

PROJECT = "landcover-riau"
ee.Authenticate()
ee.Initialize(project=PROJECT)


def show(image, vis, region, dims=400, label=None):
    params = dict(vis); params["region"] = region; params["dimensions"] = dims
    if label:
        print(label)
    display(IPyImage(url=image.getThumbURL(params)))

## 1. Class scheme + a reusable `class_mix` helper

Same 5-class remap as Notebook 2, wrapped so I can run it on any box.

In [64]:
CLASSES = {0: "forest", 1: "agriculture", 2: "urban", 3: "water", 4: "bare"}
CLASS_COLORS = ["#1b7837", "#e6c200", "#d7191c", "#2c7fb8", "#bdbdbd"]
NATIVE = [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100]
TO5    = [ 0,  1,  1,  1,  2,  4,  4,  3,  1,  0,   4]


def labels_for(region):
    wc = ee.ImageCollection("ESA/WorldCover/v200").first().clip(region)
    return wc.select("Map").remap(NATIVE, TO5).rename("class")


def class_mix(region, scale=30):
    """Return {class_name: percent} for a region's WorldCover mix."""
    hist = labels_for(region).reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=region, scale=scale, maxPixels=1e10,
    ).getInfo()["class"]
    total = sum(hist.values())
    return {CLASSES[int(k)]: 100 * v / total for k, v in hist.items()}

## 2. Candidate boxes

Each is ~0.5° × 0.5° (similar size to Kampar). They march west and north toward Pekanbaru and the Rokan palm belt: areas with more established plantation, cropland, and built-up land. `A` is the original for reference.

In [65]:
# name -> [W, S, E, N]
CANDIDATES = {
    "A_kampar_current":  [101.5,  0.0,  102.0, 0.5],   # 89.6% forest (reference)
    "F_pekanbaru_metro": [101.3,  0.35, 101.8, 0.85],  # city + agricultural fringe
    "G_indragiri_coast": [102.9, -0.6,  103.4, -0.1],  # coastal smallholder mosaic + water
    "H_bangkinang_west": [100.9,  0.2,  101.4, 0.7],   # inland agricultural belt
    "I_gradient_big":    [101.0,  0.0,  102.0, 0.7],   # larger box: forest -> plantation -> town
}

## 3. Score them

I rank by agriculture share, and flag boxes whose forest sits in the healthy 40–75% window.

In [66]:
import pandas as pd

rows = {name: class_mix(ee.Geometry.Rectangle(bbox)) for name, bbox in CANDIDATES.items()}
order = ["forest", "agriculture", "urban", "water", "bare"]
df = pd.DataFrame(rows).T.reindex(columns=order).fillna(0).round(1)
df["forest_ok"] = df["forest"].between(50, 80)

df.sort_values(["forest_ok", "agriculture"], ascending=False)

,forest,agriculture,urban,water,bare,forest_ok
G_indragiri_coast,85.2,9.8,0.5,4.4,0.0,False
A_kampar_current,89.6,8.4,1.0,0.9,0.1,False
I_gradient_big,91.3,5.1,2.6,0.8,0.2,False
F_pekanbaru_metro,89.3,3.9,5.5,0.7,0.5,False
H_bangkinang_west,93.1,3.8,2.2,0.6,0.3,False


## 4. Eyeball the contenders

Must look at the actual label maps. A good AOI shows a *mosaic* (forest blocks next to plantation/cropland), not a single colour.

In [67]:
label_vis = {"min": 0, "max": 4, "palette": CLASS_COLORS}
false_color = {"bands": ["B8", "B4", "B3"], "min": 0, "max": 3000}

for name, bbox in CANDIDATES.items():
    region = ee.Geometry.Rectangle(bbox)
    show(labels_for(region), label_vis, region=region, label=f"{name}  {bbox}")

A_kampar_current  [101.5, 0.0, 102.0, 0.5]


F_pekanbaru_metro  [101.3, 0.35, 101.8, 0.85]


G_indragiri_coast  [102.9, -0.6, 103.4, -0.1]


H_bangkinang_west  [100.9, 0.2, 101.4, 0.7]


I_gradient_big  [101.0, 0.0, 102.0, 0.7]


## 5. Pick & findings

From the table and the maps, choose the box with the best forest↔agriculture balance and a real mosaic. Set it below to preview the imagery + labels together, then write the winning bounds into `configs/config.py`.

> WorldCover counts mature oil palm as 'Tree cover' (forest), so even the "forest" share hides plantation. A box that already shows decent *agriculture* almost certainly has active clearing nearby, which is exactly what I want to detect.

In [69]:
CHOSEN = "G_indragiri_coast"   # <-- set to the winner after reading the table/maps

bbox = CANDIDATES[CHOSEN]
region = ee.Geometry.Rectangle(bbox)
print(CHOSEN, bbox)
print({k: round(v, 1) for k, v in class_mix(region).items()})

# build a quick 2021 composite just to see imagery vs labels for the pick
def s2_composite(year, reg, window=("06-01", "09-30"), threshold=0.6):
    start, end = f"{year}-{window[0]}", f"{year}-{window[1]}"
    s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(reg).filterDate(start, end))
    cs = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
    s2 = (s2.linkCollection(cs, ["cs_cdf"])
            .map(lambda i: i.updateMask(i.select("cs_cdf").gte(threshold))))
    return s2.median().clip(reg)

show(s2_composite(2021, region), false_color, region=region, dims=500, label="S2 false colour 2021")
show(labels_for(region), label_vis, region=region, dims=500, label="WorldCover -> 5 classes")

G_indragiri_coast [102.9, -0.6, 103.4, -0.1]
{'forest': 85.2, 'agriculture': 9.8, 'urban': 0.5, 'water': 4.4, 'bare': 0.0}
S2 false colour 2021


WorldCover -> 5 classes
